# 将棋勝敗予測モデル

対局前の事前情報（棋士勝率・持ち時間・戦型）のみを特徴量として、  
LightGBMで先手/後手の勝敗を予測する二値分類モデルを構築する。

**データ**: 将棋DB2より取得したKIFファイル（9棋士・約450局）  
**ターゲット**: 先手勝ち=1 / 後手勝ち=0  
**評価指標**: AUC・Accuracy（Stratified 5-fold CV）


## 1. ライブラリ・設定

In [ ]:
import sys, re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import shap
import optuna
from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.preprocessing import LabelEncoder

matplotlib.rcParams['font.family'] = 'Hiragino Sans'
optuna.logging.set_verbosity(optuna.logging.WARNING)

sys.path.insert(0, "../src")
from kif_parser import build_game_df

RANDOM_STATE = 42


## 2. データ読み込み・パース

In [ ]:
INVALID_PLAYERS = {"SUNTORY将棋オールスター東西", "深浦康市空談"}
MIN_GAMES = 3  # 勝率計算の最低対局数

def normalize_player(name: str) -> str:
    """棋士名から段位・称号を除去して姓名のみに統一"""
    if not isinstance(name, str):
        return name
    name = name.replace('\u3000', ' ').strip()
    pattern = (
        r'[\s　]*('
        r'十七世名人|竜王・名人'
        r'|[一二三四五六七八九十百]+[段冠]'
        r'|[１-９1-9][段冠]'
        r'|叡王|竜王|名人|棋聖|王位|王将|王座|棋王'
        r'|NHK杯|ＪＴ杯覇者|覇者'
        r')+'
    )
    name = re.sub(pattern, '', name).strip()
    name = re.sub(r'[\s　]+', '', name)
    return name

df = build_game_df("../data/raw")
df["sente_norm"] = df["sente"].apply(normalize_player)
df["gote_norm"]  = df["gote"].apply(normalize_player)

mask = df["sente_norm"].isin(INVALID_PLAYERS) | df["gote_norm"].isin(INVALID_PLAYERS)
df = df[~mask].reset_index(drop=True)

print(f"対局数: {len(df)}")
print(f"winner分布:\n{df['winner'].value_counts()}")
df.head()


## 3. EDA

In [ ]:
# 棋士別対局数・勝率
sente_records = df.assign(player=df["sente_norm"], win=(df["winner"]=="sente").astype(int))[["player","win"]]
gote_records  = df.assign(player=df["gote_norm"],  win=(df["winner"]=="gote").astype(int))[["player","win"]]
records = pd.concat([sente_records, gote_records], ignore_index=True)

stats = records.groupby("player")["win"].agg(games="count", wins="sum").reset_index()
stats["winrate"] = stats["wins"] / stats["games"]
stats_main = stats[stats["games"] >= 10].sort_values("winrate", ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 対局数
stats_main.sort_values("games", ascending=True).plot.barh(
    x="player", y="games", ax=axes[0], legend=False, color="steelblue"
)
axes[0].set_title("棋士別対局数（10局以上）")
axes[0].set_xlabel("対局数")

# 勝率
stats_main.sort_values("winrate", ascending=True).plot.barh(
    x="player", y="winrate", ax=axes[1], legend=False, color="coral"
)
axes[1].axvline(0.5, color="gray", linestyle="--", linewidth=1)
axes[1].set_title("棋士別勝率（10局以上）")
axes[1].set_xlabel("勝率")

plt.tight_layout()
plt.savefig("../images/eda_player_stats.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# 戦型分布
opening_counts = df["opening"].fillna("不明").value_counts()

plt.figure(figsize=(10, 5))
opening_counts.sort_values().plot.barh(color="mediumseagreen")
plt.title("戦型別対局数")
plt.xlabel("対局数")
plt.tight_layout()
plt.savefig("../images/eda_opening.png", dpi=150, bbox_inches="tight")
plt.show()


## 4. 特徴量エンジニアリング

In [ ]:
# 勝敗レコードを棋士単位に展開
sente_df = df[["sente_norm","gote_norm","winner"]].copy()
sente_df["player"]   = sente_df["sente_norm"]
sente_df["opponent"] = sente_df["gote_norm"]
sente_df["win"]      = (sente_df["winner"] == "sente").astype(int)

gote_df = df[["sente_norm","gote_norm","winner"]].copy()
gote_df["player"]   = gote_df["gote_norm"]
gote_df["opponent"] = gote_df["sente_norm"]
gote_df["win"]      = (gote_df["winner"] == "gote").astype(int)

records = pd.concat([
    sente_df[["player","opponent","win"]],
    gote_df[["player","opponent","win"]]
], ignore_index=True)

# 棋士別勝率（MIN_GAMES未満はNaN）
stats = records.groupby("player")["win"].agg(games="count", wins="sum").reset_index()
stats["winrate"] = np.where(
    stats["games"] >= MIN_GAMES,
    stats["wins"] / stats["games"],
    np.nan
)
winrate_map = stats.set_index("player")["winrate"].to_dict()
games_map   = stats.set_index("player")["games"].to_dict()

# 直接対決勝率（MIN_GAMES未満はNaN）
h2h = records.groupby(["player","opponent"])["win"].agg(
    h2h_games="count", h2h_wins="sum"
).reset_index()
h2h["h2h_winrate"] = np.where(
    h2h["h2h_games"] >= MIN_GAMES,
    h2h["h2h_wins"] / h2h["h2h_games"],
    np.nan
)
h2h_map = h2h.set_index(["player","opponent"])[["h2h_winrate","h2h_games"]].to_dict("index")

# 特徴量付与
df["sente_winrate"] = df["sente_norm"].map(winrate_map)
df["gote_winrate"]  = df["gote_norm"].map(winrate_map)
df["winrate_diff"]  = df["sente_winrate"] - df["gote_winrate"]
df["sente_games"]   = df["sente_norm"].map(games_map)
df["gote_games"]    = df["gote_norm"].map(games_map)
df["h2h_sente_winrate"] = df.apply(
    lambda r: h2h_map.get((r["sente_norm"], r["gote_norm"]), {}).get("h2h_winrate", np.nan), axis=1
)
df["h2h_games"] = df.apply(
    lambda r: h2h_map.get((r["sente_norm"], r["gote_norm"]), {}).get("h2h_games", 0), axis=1
)

# 戦型エンコード・持ち時間補完
le = LabelEncoder()
df["opening_encoded"] = le.fit_transform(df["opening"].fillna("不明"))
df["time_limit_minutes"] = df["time_limit_minutes"].fillna(df["time_limit_minutes"].median())

# ターゲット
df["target"] = (df["winner"] == "sente").astype(int)

FEATURE_COLS = [
    "sente_winrate", "gote_winrate", "winrate_diff",
    "sente_games", "gote_games",
    "h2h_games", "opening_encoded", "time_limit_minutes"
]

X = df[FEATURE_COLS]
y = df["target"]

print("欠損数:")
print(X.isnull().sum())
print(f"\nshape: {X.shape}")


## 5. LightGBM Baseline

**注意**: 勝率特徴量は全データから集計しているため情報リークが存在する。  
本モデルはポートフォリオ用のプロトタイプとして位置づけ、  
本番運用時は時系列splitによるリーク防止が必要。


In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
oof_preds_base = np.zeros(len(y))
acc_list, auc_list = [], []

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
    model = LGBMClassifier(
        n_estimators=300, learning_rate=0.05, num_leaves=31,
        random_state=RANDOM_STATE, verbose=-1
    )
    model.fit(X.iloc[tr_idx], y.iloc[tr_idx])
    prob = model.predict_proba(X.iloc[va_idx])[:, 1]
    oof_preds_base[va_idx] = prob
    acc_list.append(accuracy_score(y.iloc[va_idx], (prob >= 0.5).astype(int)))
    auc_list.append(roc_auc_score(y.iloc[va_idx], prob))
    print(f"  fold{fold+1}  acc={acc_list[-1]:.4f}  auc={auc_list[-1]:.4f}")

print(f"\nBaseline CV  acc={np.mean(acc_list):.4f}±{np.std(acc_list):.4f}"
      f"  auc={np.mean(auc_list):.4f}±{np.std(auc_list):.4f}")


## 6. Optunaハイパーパラメータチューニング

In [ ]:
def objective(trial):
    params = {
        "n_estimators":      trial.suggest_int("n_estimators", 100, 1000),
        "learning_rate":     trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "num_leaves":        trial.suggest_int("num_leaves", 16, 128),
        "max_depth":         trial.suggest_int("max_depth", 3, 10),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 50),
        "subsample":         trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree":  trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha":         trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
        "reg_lambda":        trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
        "random_state": RANDOM_STATE, "verbose": -1,
    }
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    aucs = []
    for tr_idx, va_idx in skf.split(X, y):
        m = LGBMClassifier(**params)
        m.fit(X.iloc[tr_idx], y.iloc[tr_idx])
        aucs.append(roc_auc_score(y.iloc[va_idx], m.predict_proba(X.iloc[va_idx])[:, 1]))
    return np.mean(aucs)

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100, show_progress_bar=True)
print(f"\nBest AUC: {study.best_value:.4f}")
print("Best params:", study.best_params)


In [ ]:
best_params = {**study.best_params, "random_state": RANDOM_STATE, "verbose": -1}
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
oof_preds_tuned = np.zeros(len(y))
acc_list, auc_list = [], []

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
    m = LGBMClassifier(**best_params)
    m.fit(X.iloc[tr_idx], y.iloc[tr_idx])
    prob = m.predict_proba(X.iloc[va_idx])[:, 1]
    oof_preds_tuned[va_idx] = prob
    acc_list.append(accuracy_score(y.iloc[va_idx], (prob >= 0.5).astype(int)))
    auc_list.append(roc_auc_score(y.iloc[va_idx], prob))
    print(f"  fold{fold+1}  acc={acc_list[-1]:.4f}  auc={auc_list[-1]:.4f}")

print(f"\nTuned CV  acc={np.mean(acc_list):.4f}±{np.std(acc_list):.4f}"
      f"  auc={np.mean(auc_list):.4f}±{np.std(auc_list):.4f}")


## 7. SHAP分析

In [ ]:
# 全データでモデル学習
final_model = LGBMClassifier(**best_params)
final_model.fit(X, y)

explainer = shap.TreeExplainer(final_model)
shap_values = explainer.shap_values(X)
sv = shap_values[1] if isinstance(shap_values, list) else shap_values

# Feature Importance
plt.figure()
shap.summary_plot(sv, X, feature_names=FEATURE_COLS, plot_type="bar", show=False)
plt.title("SHAP Feature Importance")
plt.tight_layout()
plt.savefig("../images/shap_importance.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# Summary Plot
plt.figure()
shap.summary_plot(sv, X, feature_names=FEATURE_COLS, show=False)
plt.title("SHAP Summary Plot")
plt.tight_layout()
plt.savefig("../images/shap_summary.png", dpi=150, bbox_inches="tight")
plt.show()


## 8. 考察

### モデル性能
| | Accuracy | AUC |
|---|---|---|
| Baseline | 0.6950±0.0408 | 0.7612±0.0296 |
| Optuna Tuned | 0.6904±0.0465 | 0.7807±0.0363 |

- 事前情報のみでAUC=0.78はランダム(0.5)を大きく上回る
- `max_depth=3`が選ばれた → データ規模に対してシンプルなモデルが適切

### SHAP考察
- **winrate_diff**が断トツ1位 → 実力差が勝敗を最も強く規定する（将棋的に直感と一致）
- **gote_winrate**が単独でも効く → `winrate_diff`との多重共線性による影響の可能性あり
- **time_limit_minutes** → 持ち時間による棋風差・コンディション差が一定程度影響
- **h2h_games** → ほぼ寄与なし（欠損率56%のため情報が薄い）

### 限界・今後の課題
- 勝率特徴量は全データ集計のため**情報リークあり**。時系列splitによる厳密評価が必要
- サンプル数449局は機械学習としては小規模。追加データ取得で精度改善の余地あり
- エンジン評価値が取得できれば局面途中予測モデル（Phase 2）への拡張が可能
